# Experiment 1 · Notebook 1 — Load and Join

**Goal:** Build a clean training set joining OOTP power ratings (the labels) to real-world career batting stats (the features), filtered to MLB-experienced hitters and feature-engineered for modeling.

**Output:** `data/training_data.csv` with one row per hitter:
`HistID, Name, POW, bats, hr_per_600, bats_L, bats_S, HR, PA, source`.

## 1. Setup

We try three stat sources in `auto` mode:

1. **MLB Stats API** (`statsapi.mlb.com`) — official, free, current. Default.
2. **pybaseball** (FanGraphs) — comprehensive but FanGraphs frequently    blocks the scraper.
3. **Lahman** (`data/Batting.csv`) — bundled, capped at 2021. Fallback only.

Run all the cells in order; the cache files keep reruns fast.

In [43]:
from __future__ import annotations

import json
import time
from pathlib import Path
from typing import Iterable

import pandas as pd

# Paths (notebook expects to be run from experiment1/)
OOTP_PATH        = Path("../../data/ootp_ratings/ootp_batting_ratings.csv")
LAHMAN_PATH      = Path("data/Batting.csv")
OUTPUT_PATH      = Path("data/training_data.csv")
MLB_CACHE_PATH   = Path(".mlb_career_cache.json")
PYBASEBALL_CACHE = Path(".pybaseball_cache")

# Config
MLB_STATS_API        = "https://statsapi.mlb.com/api/v1/people"
MLB_BATCH_SIZE       = 50
MLB_REQUEST_TIMEOUT  = 30
MLB_REQUEST_PAUSE    = 0.2
PA_THRESHOLD         = 500

# Choose the source: 'mlb', 'pybaseball', 'lahman', or 'auto'
SOURCE = "auto"


## 2. Load OOTP labels

We only need four columns: the bbref join key, the player name (for sanity checks), the POW rating (target), and batter handedness.

In [44]:
ootp = pd.read_csv(OOTP_PATH)
ootp = ootp[["HistID", "Name", "POW", "B_1"]].dropna(subset=["HistID"]).copy()
ootp["HistID"] = ootp["HistID"].astype(str).str.strip()

print(f"OOTP players: {len(ootp)}")
print(f"Bats: {dict(ootp['B_1'].value_counts())}")
print(f"POW: mean={ootp['POW'].mean():.1f}, std={ootp['POW'].std():.1f}, range={ootp['POW'].min()}-{ootp['POW'].max()}")
ootp.head()


OOTP players: 667
Bats: {'R': np.int64(364), 'L': np.int64(232), 'S': np.int64(71)}
POW: mean=45.9, std=11.5, range=20-80


,HistID,Name,POW,B_1
0,judgeaa01,Aaron Judge,80,R
1,schunaa01,Aaron Schunk,40,R
2,toroab01,Abraham Toro,40,S
3,amadoad01,Adael Amador,30,S
4,fraziad01,Adam Frazier,35,L


## 3. MLB Stats API — career hitting helpers

We:
1. Map bbref IDs → MLBAM IDs via `pybaseball.playerid_reverse_lookup` (this only hits the Chadwick Bureau registry, which is reliable).
2. Batch-call `/api/v1/people` with the career-hitting hydration.
3. Cache responses to `.mlb_career_cache.json` so reruns are instant.

In [35]:
def _load_mlb_cache() -> dict[str, dict]:
    if MLB_CACHE_PATH.exists():
        try:
            return json.loads(MLB_CACHE_PATH.read_text())
        except json.JSONDecodeError:
            pass
    return {}

def _save_mlb_cache(cache: dict[str, dict]) -> None:
    MLB_CACHE_PATH.write_text(json.dumps(cache, indent=0))

def _batched(seq: list, n: int) -> Iterable[list]:
    for i in range(0, len(seq), n):
        yield seq[i:i + n]

def _extract_career_hitting(person: dict) -> tuple[int, int]:
    """Pull HR / PA out of a /people response."""
    hr = pa = 0
    for stat_block in person.get("stats", []) or []:
        if stat_block.get("group", {}).get("displayName", "").lower() != "hitting":
            continue
        if stat_block.get("type", {}).get("displayName", "").lower() != "career":
            continue
        for split in stat_block.get("splits", []) or []:
            stat = split.get("stat", {})
            hr += int(stat.get("homeRuns", 0) or 0)
            pa += int(stat.get("plateAppearances", 0) or 0)
    return hr, pa


In [36]:
def load_career_via_mlb_api(bbref_ids: list[str]) -> pd.DataFrame:
    import requests
    from pybaseball import cache as pb_cache, playerid_reverse_lookup

    PYBASEBALL_CACHE.mkdir(exist_ok=True)
    pb_cache.config.cache_directory = str(PYBASEBALL_CACHE)
    pb_cache.enable()

    print(f"Mapping {len(bbref_ids)} bbref IDs -> MLBAM IDs ...")
    id_map = playerid_reverse_lookup(bbref_ids, key_type="bbref")
    id_map = id_map.dropna(subset=["key_mlbam"]).copy()
    id_map["key_mlbam"] = id_map["key_mlbam"].astype(int)
    print(f"Mapped {len(id_map)} / {len(bbref_ids)}")

    cache_data = _load_mlb_cache()
    mlb_ids = id_map["key_mlbam"].astype(str).tolist()
    missing = [m for m in mlb_ids if m not in cache_data]
    print(f"Cache hits: {len(mlb_ids) - len(missing)} / {len(mlb_ids)}; fetching {len(missing)}")

    session = requests.Session()
    session.headers.update({"User-Agent": "experiment1-pow/1.0"})

    for i, batch in enumerate(_batched(missing, MLB_BATCH_SIZE), start=1):
        params = {
            "personIds": ",".join(batch),
            "hydrate": "stats(group=[hitting],type=[career])",
        }
        r = session.get(MLB_STATS_API, params=params, timeout=MLB_REQUEST_TIMEOUT)
        r.raise_for_status()
        for person in (r.json().get("people", []) or []):
            mlb_id = str(person["id"])
            hr, pa = _extract_career_hitting(person)
            cache_data[mlb_id] = {"HR": hr, "PA": pa}
        if i % 5 == 0 or i == 1:
            print(f"  fetched batch {i} ({i * MLB_BATCH_SIZE} ids)")
            _save_mlb_cache(cache_data)
        time.sleep(MLB_REQUEST_PAUSE)
    _save_mlb_cache(cache_data)

    rows = []
    for _, row in id_map.iterrows():
        entry = cache_data.get(str(row["key_mlbam"]))
        if entry is None:
            continue
        rows.append({
            "playerID": row["key_bbref"],
            "HR": int(entry.get("HR", 0)),
            "PA": int(entry.get("PA", 0)),
        })
    career = pd.DataFrame(rows)
    return career[career["PA"] > 0].reset_index(drop=True)


## 4. Pull career stats

First run hits `statsapi.mlb.com` ~14 times (50 IDs per batch) — about a minute. Subsequent runs read from `.mlb_career_cache.json` and finish in seconds.

In [37]:
bbref_ids = ootp["HistID"].dropna().unique().tolist()

career = None
source_used = None

sources_to_try = ["mlb", "pybaseball", "lahman"] if SOURCE == "auto" else [SOURCE]

for src in sources_to_try:
    try:
        if src == "mlb":
            print("\n>>> MLB Stats API")
            career = load_career_via_mlb_api(bbref_ids)
            source_used = "mlb_api"
        elif src == "pybaseball":
            print("\n>>> pybaseball / FanGraphs (1985-2024)")
            from pybaseball import batting_stats, cache as pb_cache, playerid_reverse_lookup
            PYBASEBALL_CACHE.mkdir(exist_ok=True)
            pb_cache.config.cache_directory = str(PYBASEBALL_CACHE)
            pb_cache.enable()
            id_map = playerid_reverse_lookup(bbref_ids, key_type="bbref")
            id_map = id_map.dropna(subset=["key_fangraphs"]).copy()
            id_map["key_fangraphs"] = id_map["key_fangraphs"].astype(int)
            fg = batting_stats(1985, 2024, qual=0)
            fg["IDfg"] = fg["IDfg"].astype(int)
            fg = fg[fg["IDfg"].isin(id_map["key_fangraphs"])]
            fg = fg.merge(id_map[["key_bbref", "key_fangraphs"]],
                          left_on="IDfg", right_on="key_fangraphs", how="inner")
            career = (fg.groupby("key_bbref", as_index=False)
                        .agg(HR=("HR", "sum"), PA=("PA", "sum"))
                        .rename(columns={"key_bbref": "playerID"}))
            source_used = "pybaseball_1985_2024"
        else:
            print("\n>>> Lahman fallback")
            lahman = pd.read_csv(LAHMAN_PATH)
            cols = ["AB", "HR", "BB", "HBP", "SH", "SF"]
            lahman[cols] = lahman[cols].fillna(0).astype(int)
            agg = (lahman.groupby('playerID', as_index=False)
                         .agg(HR=('HR','sum'), AB=('AB','sum'), BB=('BB','sum'),
                              HBP=('HBP','sum'), SF=('SF','sum'), SH=('SH','sum')))
            agg["PA"] = agg["AB"] + agg["BB"] + agg["HBP"] + agg["SF"] + agg["SH"]
            career = agg[["playerID", "HR", "PA"]]
            max_year = pd.read_csv(LAHMAN_PATH, usecols=["yearID"])["yearID"].max()
            source_used = f"lahman_through_{max_year}"
        break
    except Exception as e:
        print(f"  source {src!r} failed: {e}")
        if len(sources_to_try) == 1:
            raise

print(f"\nSource used: {source_used}")
print(f"Career rows: {len(career)}")
career.head()



>>> MLB Stats API
Mapping 667 bbref IDs -> MLBAM IDs ...
Mapped 667 / 667
Cache hits: 667 / 667; fetching 0

Source used: mlb_api
Career rows: 667


,playerID,HR,PA
0,milleow01,15,1032
1,edmanto01,72,2955
2,schneda03,33,885
3,durbica01,12,620
4,lopezni01,7,2378


## 5. Join OOTP labels to career stats

Inner join on `HistID` ↔ `playerID`. Drop hitters with fewer than 500 career PA — below that threshold, OOTP's POW rating leans heavily on scouting projections, which adds noise to the relationship we're trying to learn.

In [38]:
merged = ootp.merge(career, left_on="HistID", right_on="playerID", how="inner")
print(f"Joined: {len(merged):>4} / {len(ootp)} OOTP players matched")

df = merged[merged["PA"] >= PA_THRESHOLD].copy()
print(f"After PA >= {PA_THRESHOLD}: {len(df):>4} players (dropped {len(merged) - len(df)})")


Joined:  667 / 667 OOTP players matched
After PA >= 500:  441 players (dropped 226)


## 6. Feature engineering

Two features:
- `hr_per_600 = (HR / PA) * 600` — career HR rate, normalized to a   full-season cadence so workload doesn't dominate.
- One-hot handedness with R as the reference: `bats_L` and `bats_S`.

In [39]:
df["hr_per_600"] = (df["HR"] / df["PA"]) * 600
df["bats"] = df["B_1"]
df["bats_L"] = (df["B_1"] == "L").astype(int)
df["bats_S"] = (df["B_1"] == "S").astype(int)
df["source"] = source_used

cols = [
    "HistID", "Name", "POW", "bats",
    "hr_per_600", "bats_L", "bats_S",
    "HR", "PA", "source",
]
training = df[cols].sort_values("hr_per_600", ascending=False).reset_index(drop=True)
training.head()


,HistID,Name,POW,bats,hr_per_600,bats_L,bats_S,HR,PA,source
0,judgeaa01,Aaron Judge,80,R,44.314869,0,0,380,5145,mlb_api
1,kurtzni01,Nick Kurtz,80,L,38.862559,1,0,41,633,mlb_api
2,ohtansh01,Shohei Ohtani,80,L,38.363514,1,0,286,4473,mlb_api
3,schwaky01,Kyle Schwarber,80,L,38.096961,1,0,351,5528,mlb_api
4,stantmi03,Giancarlo Stanton,80,R,37.618589,0,0,456,7273,mlb_api


## 7. Save

Write the cleaned training set; notebook 2 picks up from here.

In [40]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
training.to_csv(OUTPUT_PATH, index=False)
print(f"Wrote {len(training)} rows -> {OUTPUT_PATH}")

print("\nTop 5 by HR/600 PA:")
print(training.head().to_string(index=False))
print("\nBottom 5 by HR/600 PA:")
print(training.tail().to_string(index=False))


Wrote 441 rows -> data/training_data.csv

Top 5 by HR/600 PA:
   HistID              Name  POW bats  hr_per_600  bats_L  bats_S  HR   PA  source
judgeaa01       Aaron Judge   80    R   44.314869       0       0 380 5145 mlb_api
kurtzni01        Nick Kurtz   80    L   38.862559       1       0  41  633 mlb_api
ohtansh01     Shohei Ohtani   80    L   38.363514       1       0 286 4473 mlb_api
schwaky01    Kyle Schwarber   80    L   38.096961       1       0 351 5528 mlb_api
stantmi03 Giancarlo Stanton   80    R   37.618589       0       0 456 7273 mlb_api

Bottom 5 by HR/600 PA:
   HistID             Name  POW bats  hr_per_600  bats_L  bats_S  HR   PA  source
guilllu01   Luis Guillorme   20    L    2.909796       1       0   5 1031 mlb_api
edwarxa01   Xavier Edwards   25    S    2.622378       0       1   5 1144 mlb_api
  baeji01      Ji-hwan Bae   35    L    2.334630       1       0   2  514 mlb_api
lopezni01      Nicky Lopez   20    L    1.766190       1       0   7 2378 mlb_api
simpsc